# XL-LEXEME Semantic Breadth

This notebook estimates annual semantic breadth for ADHD, Autism, and the three baseline terms. Breadth is operationalised as within-year contextual dispersion among target-aware XL-LEXEME embeddings. The notebook keeps the existing marking, sampling, embedding, and cosine-distance logic, but now reports target estimates by substantive frame stratum.


## Setup

The diachronic axis is publication year (`lsc_year`). Target contexts are restricted to the three core substantive frames and duplicated into `substantive_core_overall`; baselines remain unframed. The current execution uses all markable target contexts and caps only baseline samples at 1,000 contexts per baseline-year.


In [1]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 180)
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#263238",
        "axes.labelcolor": "#263238",
        "axes.titlecolor": "#263238",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#D7DEE2",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.7,
        "font.family": "DejaVu Sans",
        "font.size": 10.5,
        "legend.frameon": False,
        "xtick.color": "#263238",
        "ytick.color": "#263238",
    }
)


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "configs/commoncrawl_collection.yaml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not locate repository root from current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
CONTEXT_PATH = PROJECT_ROOT / "data/interim/lsc/contexts/lsc_mention_contexts.parquet"
FRAME_LABEL_PATH = PROJECT_ROOT / "data/processed/lsc/classification/lsc_target_context_frame_labels.csv"
INTERIM_DIR = PROJECT_ROOT / "data/interim/lsc/breadth"
PROCESSED_DIR = PROJECT_ROOT / "data/processed/lsc/breadth"
FIGURE_DIR = PROJECT_ROOT / "reports/figures/lsc/breadth"
for directory in [INTERIM_DIR, PROCESSED_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MODEL_PATH_CANDIDATES = [
    PROJECT_ROOT / "data/external/models/xl-lexeme",
    PROJECT_ROOT / "data/external/model",
]
XL_LEXEME_MODEL_PATH = next((path for path in MODEL_PATH_CANDIDATES if path.exists()), MODEL_PATH_CANDIDATES[0])

EXPECTED_YEARS = list(range(2014, 2027))
TARGET_UNITS = ["ADHD", "Autism"]
BASELINE_UNITS = ["frustration", "loneliness", "sadness"]
EXPECTED_UNITS = TARGET_UNITS + BASELINE_UNITS
CORE_TARGET_FRAMES = ["clinical_only", "lived_only", "mixed"]
TARGET_FRAME_STRATA = ["substantive_core_overall", *CORE_TARGET_FRAMES]
BASELINE_FRAME_STRATUM = "unframed_baseline"

UNIT_COLOURS = {
    "ADHD": "#2F6F9F",
    "Autism": "#B66A4A",
    "frustration": "#4F8F78",
    "loneliness": "#7FA68A",
    "sadness": "#9AA6A1",
}
UNIT_MARKERS = {
    "ADHD": "o",
    "Autism": "s",
    "frustration": "^",
    "loneliness": "D",
    "sadness": "v",
}
FRAME_LABELS = {
    "substantive_core_overall": "Overall",
    "clinical_only": "Clinical/disorder framing",
    "lived_only": "Lived-experience framing",
    "mixed": "Mixed clinical/lived framing",
    "unframed_baseline": "Comparator term",
}
FRAME_COLORS = {
    "substantive_core_overall": "#263238",
    "clinical_only": "#4F8DB3",
    "lived_only": "#C98263",
    "mixed": "#79A889",
    "substantive_other": "#A998C9",
    "non_substantive_or_insufficient": "#B8C0C5",
    "unframed_baseline": "#7B8785",
}
CONDITION_FRAME_COLORS = {
    "ADHD": {
        "substantive_core_overall": "#2F6F9F",
        "clinical_only": "#75A9C8",
        "lived_only": "#AECFE0",
        "mixed": "#D4E4EC",
    },
    "Autism": {
        "substantive_core_overall": "#B66A4A",
        "clinical_only": "#CE8D70",
        "lived_only": "#E1B49D",
        "mixed": "#F2D8CF",
    },
}
FRAME_MARKERS = {
    "substantive_core_overall": "o",
    "clinical_only": "s",
    "lived_only": "^",
    "mixed": "D",
    "unframed_baseline": "o",
}

TARGET_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME: int | None = None
BASELINE_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME: int | None = 1000
RANDOM_SEED = 123
BOOTSTRAP_REPETITIONS = 500
MIN_CONTEXT_TOKENS = 8
MAX_SEQUENCE_LENGTH = 128
ENCODE_BATCH_SIZE = 8
DEVICE = "cpu"
TARGET_START = "<t>"
TARGET_END = "</t>"
REBUILD_XL_LEXEME_EMBEDDINGS = False
TOKEN_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)
MIN_FRAME_CONTEXTS_FOR_INTERPRETATION = 100
MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION = 50
DW_AUTOCORRELATION_LOW = 1.25
DW_AUTOCORRELATION_HIGH = 2.75
LSC_FIGURE_DPI = 300
SAMPLING_RNG = np.random.default_rng(RANDOM_SEED)
BOOTSTRAP_RNG = np.random.default_rng(RANDOM_SEED + 1)


/opt/anaconda3/envs/msc-nlp/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Shared Contexts And Join Frame Labels

The stable frame-classifier `context_id` is reconstructed from the same fields used by the classifier application notebook. Only target contexts in `clinical_only`, `lived_only`, or `mixed` enter semantic breadth estimates; those rows are sampled both within their frame and within `substantive_core_overall`.


In [2]:
if not CONTEXT_PATH.exists():
    raise FileNotFoundError(f"Missing shared LSC context table: {CONTEXT_PATH}")
if not FRAME_LABEL_PATH.exists():
    raise FileNotFoundError(f"Missing frame-label handoff: {FRAME_LABEL_PATH}")
if not XL_LEXEME_MODEL_PATH.exists():
    raise FileNotFoundError(
        "Missing local XL-LEXEME model. Expected one of: "
        + ", ".join(str(path) for path in MODEL_PATH_CANDIDATES)
    )

context_columns = [
    "doc_id",
    "lsc_year",
    "published_year",
    "source_year",
    "analysis_unit",
    "term_role",
    "target_group",
    "raw_form",
    "matched_text",
    "mention_start_char",
    "mention_end_char",
    "collapsed_matched_texts",
    "registered_domain",
    "target_sentence",
    "target_sentence_plus_adjacent",
]
contexts = pd.read_parquet(CONTEXT_PATH, columns=context_columns).reset_index(drop=True)
contexts["source_context_row_id"] = contexts.index.astype(int)
contexts["registered_domain"] = contexts["registered_domain"].fillna("unknown_domain")
contexts["target_group"] = contexts["target_group"].fillna("baseline")


def stable_context_id(row: pd.Series) -> str:
    value = "|".join(
        str(row.get(column, ""))
        for column in ["doc_id", "analysis_unit", "raw_form", "mention_start_char", "mention_end_char"]
    )
    return hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]


frame_labels = pd.read_csv(
    FRAME_LABEL_PATH,
    usecols=["context_id", "predicted_derived_frame", "p_substantive", "p_clinical_given_substantive", "p_lived_given_substantive"],
)
if frame_labels["context_id"].duplicated().any():
    raise RuntimeError("Frame-label handoff contains duplicate context IDs.")

target_mask = contexts["analysis_unit"].isin(TARGET_UNITS)
contexts.loc[target_mask, "context_id"] = contexts.loc[target_mask].apply(stable_context_id, axis=1)
contexts = contexts.merge(frame_labels, on="context_id", how="left")
missing_target_labels = contexts.loc[target_mask, "predicted_derived_frame"].isna().sum()
if missing_target_labels:
    raise RuntimeError(f"Missing frame labels for {missing_target_labels:,} target contexts.")

baseline_contexts = contexts.loc[~target_mask].copy()
baseline_contexts["frame_stratum"] = BASELINE_FRAME_STRATUM
core_target_contexts = contexts.loc[
    target_mask & contexts["predicted_derived_frame"].isin(CORE_TARGET_FRAMES)
].copy()
target_by_frame = core_target_contexts.copy()
target_by_frame["frame_stratum"] = target_by_frame["predicted_derived_frame"]
target_overall = core_target_contexts.copy()
target_overall["frame_stratum"] = "substantive_core_overall"

analysis_contexts = pd.concat([baseline_contexts, target_overall, target_by_frame], ignore_index=True, sort=False)
analysis_contexts["context_row_id"] = np.arange(len(analysis_contexts), dtype=int)

observed_units = sorted(analysis_contexts["analysis_unit"].dropna().unique())
observed_years = sorted(analysis_contexts["lsc_year"].dropna().astype(int).unique())
missing_units = sorted(set(EXPECTED_UNITS) - set(observed_units))
missing_years = sorted(set(EXPECTED_YEARS) - set(observed_years))
if missing_units:
    raise RuntimeError(f"Missing expected analysis units: {missing_units}")
if missing_years:
    raise RuntimeError(f"Missing expected LSC years: {missing_years}")

input_summary = pd.DataFrame(
    {
        "metric": ["source_contexts", "analysis_context_rows", "documents", "analysis_units", "years", "model_path"],
        "value": [
            len(contexts),
            len(analysis_contexts),
            analysis_contexts["doc_id"].nunique(),
            ", ".join(observed_units),
            f"{min(observed_years)}-{max(observed_years)}",
            str(XL_LEXEME_MODEL_PATH.relative_to(PROJECT_ROOT)),
        ],
    }
)
input_summary


,metric,value
0,source_contexts,293670
1,analysis_context_rows,311030
2,documents,192046
3,analysis_units,"ADHD, Autism, frustration, loneliness, sadness"
4,years,2014-2026
5,model_path,data/external/models/xl-lexeme


## Mark Target Contexts

XL-LEXEME needs explicit target markers. The notebook first tries the target sentence, then falls back to target sentence plus adjacent context when the sentence is short or cannot be marked reliably. Rows that cannot be marked are saved as diagnostics and excluded from embedding.


In [3]:
def token_count(text: object) -> int:
    return len(TOKEN_RE.findall(str(text or "")))


def split_pipe_values(value: object) -> list[str]:
    if pd.isna(value):
        return []
    return [part.strip() for part in str(value).split("|") if part.strip()]


def whitespace_flexible_pattern(text: str) -> str:
    escaped = re.escape(" ".join(str(text).split()))
    return escaped.replace(r"\ ", r"\s+")


def raw_form_patterns(raw_form: str) -> list[str]:
    patterns = {
        "adhd": [r"\bADHD\b"],
        "attention_deficit": [r"\battention\s+deficit(?:\s+hyperactivity(?:\s+disorder)?)?\b"],
        "autism": [r"\bautism\b"],
        "autistic": [r"\bautistic\b"],
        "autism_spectrum": [r"\bautism\s+spectrum\b"],
        "asd_disambiguated": [r"\bASD\b"],
        "frustration": [r"\bfrustration\b"],
        "loneliness": [r"\bloneliness\b"],
        "sadness": [r"\bsadness\b"],
    }
    return patterns.get(raw_form, [r"\b" + re.escape(raw_form.replace("_", " ")) + r"\b"])


def candidate_patterns(row: pd.Series) -> list[tuple[str, str]]:
    candidates: list[tuple[str, str]] = []
    matched_values = [row.get("matched_text")] + split_pipe_values(row.get("collapsed_matched_texts"))
    for value in matched_values:
        if isinstance(value, str) and value.strip():
            candidates.append(("matched_text", whitespace_flexible_pattern(value)))
    for pattern in raw_form_patterns(str(row.get("raw_form") or "")):
        candidates.append(("raw_form", pattern))
    seen: set[str] = set()
    unique_candidates = []
    for source, pattern in candidates:
        if pattern not in seen:
            unique_candidates.append((source, pattern))
            seen.add(pattern)
    return unique_candidates


def mark_first_match(text: str, row: pd.Series) -> tuple[str | None, str | None, str | None]:
    for pattern_source, pattern in candidate_patterns(row):
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            start, end = match.span()
            marked = text[:start] + TARGET_START + text[start:end] + TARGET_END + text[end:]
            return marked, pattern_source, text[start:end]
    return None, None, None


def context_candidates(row: pd.Series) -> list[tuple[str, str]]:
    sentence = str(row.get("target_sentence") or "").strip()
    adjacent = str(row.get("target_sentence_plus_adjacent") or "").strip()
    candidates: list[tuple[str, str]] = []
    if token_count(sentence) >= MIN_CONTEXT_TOKENS:
        candidates.append(("target_sentence", sentence))
        if adjacent and adjacent != sentence:
            candidates.append(("target_sentence_plus_adjacent", adjacent))
    else:
        if adjacent:
            candidates.append(("target_sentence_plus_adjacent", adjacent))
        if sentence:
            candidates.append(("target_sentence", sentence))
    return candidates


def select_marked_context(row: pd.Series) -> dict[str, object]:
    for context_source, text in context_candidates(row):
        marked, pattern_source, marked_text = mark_first_match(text, row)
        if marked:
            return {
                "marked_context": marked,
                "context_source": context_source,
                "mark_pattern_source": pattern_source,
                "marked_text": marked_text,
                "context_token_count": token_count(text),
                "markable": True,
                "unmarkable_reason": "",
            }
    return {
        "marked_context": "",
        "context_source": "",
        "mark_pattern_source": "",
        "marked_text": "",
        "context_token_count": 0,
        "markable": False,
        "unmarkable_reason": "target_span_not_found_in_context",
    }


mark_records = [
    select_marked_context(row)
    for _, row in tqdm(analysis_contexts.iterrows(), total=len(analysis_contexts), desc="Marking target contexts")
]
mark_data = pd.DataFrame(mark_records)
marked_contexts = pd.concat([analysis_contexts, mark_data], axis=1)

unmarkable_contexts = marked_contexts.loc[~marked_contexts["markable"]].copy()
markable_contexts = marked_contexts.loc[marked_contexts["markable"]].copy()

dedupe_subset = ["analysis_unit", "lsc_year", "frame_stratum", "doc_id", "marked_context"]
markable_before_dedupe = len(markable_contexts)
markable_contexts = markable_contexts.drop_duplicates(subset=dedupe_subset).reset_index(drop=True)
duplicate_marked_contexts_removed = markable_before_dedupe - len(markable_contexts)

unmarkable_path = INTERIM_DIR / "lsc_breadth_unmarkable_contexts.csv"
unmarkable_columns = [
    "context_row_id",
    "source_context_row_id",
    "doc_id",
    "lsc_year",
    "analysis_unit",
    "frame_stratum",
    "raw_form",
    "matched_text",
    "unmarkable_reason",
]
unmarkable_contexts[unmarkable_columns].to_csv(unmarkable_path, index=False)

pd.DataFrame(
    {
        "metric": ["markable_contexts", "unmarkable_contexts", "duplicate_marked_contexts_removed"],
        "value": [len(markable_contexts), len(unmarkable_contexts), duplicate_marked_contexts_removed],
    }
)


Marking target contexts: 100%|██████████| 311030/311030 [00:11<00:00, 26171.72it/s]


,metric,value
0,markable_contexts,301601
1,unmarkable_contexts,0
2,duplicate_marked_contexts_removed,9429


## Sample Contexts

The Breadth run uses all markable ADHD and Autism substantive-core target contexts, including the separate clinical-only, lived-only, and mixed frame strata. Baseline terms remain unframed and are sampled deterministically with a 1,000-context cap per baseline-year, stratified by registered domain so that one high-volume site does not dominate the comparator trajectory.


In [4]:
def domain_stratified_sample(group: pd.DataFrame, cap: int | None, rng: np.random.Generator) -> pd.DataFrame:
    if cap is None or len(group) <= cap:
        return group.copy()
    domain_counts = group["registered_domain"].value_counts().sort_index()
    domain_names = domain_counts.index.to_numpy()
    ideal = domain_counts.to_numpy(dtype=float) / domain_counts.sum() * cap
    quotas = np.floor(ideal).astype(int)
    remainders = ideal - quotas

    while quotas.sum() < cap:
        candidates = np.where(quotas < domain_counts.to_numpy())[0]
        if len(candidates) == 0:
            break
        best = candidates[np.argmax(remainders[candidates])]
        quotas[best] += 1
        remainders[best] = -1

    sampled_parts = []
    for domain, quota in zip(domain_names, quotas):
        if quota <= 0:
            continue
        domain_rows = group[group["registered_domain"] == domain]
        sampled_index = rng.choice(domain_rows.index.to_numpy(), size=min(quota, len(domain_rows)), replace=False)
        sampled_parts.append(group.loc[sampled_index])

    sampled = pd.concat(sampled_parts, axis=0) if sampled_parts else group.iloc[0:0].copy()
    if len(sampled) < cap:
        remaining = group.drop(index=sampled.index)
        fill_count = min(cap - len(sampled), len(remaining))
        fill_index = rng.choice(remaining.index.to_numpy(), size=fill_count, replace=False)
        sampled = pd.concat([sampled, group.loc[fill_index]], axis=0)
    return sampled.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)


def sample_cap_for_group(group: pd.DataFrame) -> int | None:
    analysis_unit = group["analysis_unit"].iloc[0]
    if analysis_unit in TARGET_UNITS:
        return TARGET_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME
    return BASELINE_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME


GROUP_COLUMNS = ["analysis_unit", "lsc_year", "frame_stratum"]
available_counts = analysis_contexts.groupby(GROUP_COLUMNS, as_index=False).agg(available_contexts=("doc_id", "size"))
markable_counts = marked_contexts.loc[marked_contexts["markable"]].groupby(GROUP_COLUMNS, as_index=False).agg(markable_contexts_before_dedupe=("doc_id", "size"))
deduped_counts = markable_contexts.groupby(GROUP_COLUMNS, as_index=False).agg(markable_contexts=("doc_id", "size"))

sampled_groups = []
for _, group in markable_contexts.groupby(GROUP_COLUMNS, sort=True):
    sampled_groups.append(domain_stratified_sample(group, sample_cap_for_group(group), SAMPLING_RNG))
sampled_contexts = pd.concat(sampled_groups, axis=0).reset_index(drop=True)
sampled_contexts["sample_row_id"] = np.arange(len(sampled_contexts), dtype=int)

sampled_counts = (
    sampled_contexts.groupby(GROUP_COLUMNS, as_index=False)
    .agg(
        sampled_contexts=("doc_id", "size"),
        sampled_documents=("doc_id", "nunique"),
        sampled_domains=("registered_domain", "nunique"),
        target_sentence_contexts=("context_source", lambda values: int((values == "target_sentence").sum())),
        adjacent_contexts=("context_source", lambda values: int((values == "target_sentence_plus_adjacent").sum())),
    )
)

top_domain_share = (
    sampled_contexts.groupby([*GROUP_COLUMNS, "registered_domain"], as_index=False)
    .size()
    .rename(columns={"size": "domain_contexts"})
)
top_domain_share["sampled_contexts_for_share"] = top_domain_share.groupby(GROUP_COLUMNS)["domain_contexts"].transform("sum")
top_domain_share["domain_share"] = top_domain_share["domain_contexts"] / top_domain_share["sampled_contexts_for_share"]
top_domain_share = top_domain_share.sort_values("domain_share", ascending=False).groupby(GROUP_COLUMNS, as_index=False).head(1)
top_domain_share = top_domain_share.rename(columns={"registered_domain": "top_domain", "domain_share": "top_domain_share"})[
    [*GROUP_COLUMNS, "top_domain", "top_domain_share"]
]

sampling_diagnostics = available_counts.merge(markable_counts, on=GROUP_COLUMNS, how="left")
sampling_diagnostics = sampling_diagnostics.merge(deduped_counts, on=GROUP_COLUMNS, how="left")
sampling_diagnostics = sampling_diagnostics.merge(sampled_counts, on=GROUP_COLUMNS, how="left")
sampling_diagnostics = sampling_diagnostics.merge(top_domain_share, on=GROUP_COLUMNS, how="left")
for column in ["markable_contexts_before_dedupe", "markable_contexts", "sampled_contexts", "sampled_documents", "sampled_domains"]:
    sampling_diagnostics[column] = sampling_diagnostics[column].fillna(0).astype(int)
sampling_diagnostics["unmarkable_contexts"] = sampling_diagnostics["available_contexts"] - sampling_diagnostics["markable_contexts_before_dedupe"]
sampling_diagnostics["duplicate_marked_contexts_removed"] = sampling_diagnostics["markable_contexts_before_dedupe"] - sampling_diagnostics["markable_contexts"]
sampling_diagnostics["unmarkable_share"] = sampling_diagnostics["unmarkable_contexts"] / sampling_diagnostics["available_contexts"].replace(0, np.nan)
sampling_diagnostics["cap_applied"] = sampling_diagnostics["sampled_contexts"] < sampling_diagnostics["markable_contexts"]
sampling_diagnostics["sampling_policy"] = np.where(
    sampling_diagnostics["analysis_unit"].isin(TARGET_UNITS),
    "all_markable_target_contexts",
    "domain_stratified_baseline_cap",
)
sampling_diagnostics["max_contexts_per_unit_year_frame"] = np.where(
    sampling_diagnostics["analysis_unit"].isin(TARGET_UNITS),
    -1 if TARGET_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME is None else TARGET_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME,
    -1 if BASELINE_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME is None else BASELINE_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME,
)
sampling_diagnostics["small_cell_flag"] = sampling_diagnostics["frame_stratum"].isin(CORE_TARGET_FRAMES) & (
    sampling_diagnostics["sampled_contexts"].lt(MIN_FRAME_CONTEXTS_FOR_INTERPRETATION)
    | sampling_diagnostics["sampled_documents"].lt(MIN_FRAME_DOCUMENTS_FOR_INTERPRETATION)
)
sampling_diagnostics = sampling_diagnostics.sort_values(GROUP_COLUMNS).reset_index(drop=True)

raw_form_diagnostics = (
    sampled_contexts.groupby([*GROUP_COLUMNS, "term_role", "target_group", "raw_form"], as_index=False)
    .agg(sampled_contexts=("doc_id", "size"), sampled_documents=("doc_id", "nunique"))
)
raw_form_diagnostics["raw_form_context_share"] = raw_form_diagnostics["sampled_contexts"] / raw_form_diagnostics.groupby(GROUP_COLUMNS)["sampled_contexts"].transform("sum")

sampled_contexts_path = INTERIM_DIR / "lsc_breadth_sampled_contexts.parquet"
sampling_diagnostics_path = PROCESSED_DIR / "lsc_breadth_sampling_diagnostics.csv"
raw_form_diagnostics_path = PROCESSED_DIR / "lsc_breadth_raw_form_diagnostics.csv"
sampling_diagnostics.to_csv(sampling_diagnostics_path, index=False)
raw_form_diagnostics.to_csv(raw_form_diagnostics_path, index=False)

sampling_diagnostics.head(12)


,analysis_unit,lsc_year,frame_stratum,available_contexts,markable_contexts_before_dedupe,markable_contexts,sampled_contexts,sampled_documents,sampled_domains,target_sentence_contexts,adjacent_contexts,top_domain,top_domain_share,unmarkable_contexts,duplicate_marked_contexts_removed,unmarkable_share,cap_applied,sampling_policy,max_contexts_per_unit_year_frame,small_cell_flag
0,ADHD,2014,clinical_only,1286,1286,1220,1220,808,663,1196,24,naturalnews.com,0.021311,0,66,0.0,False,all_markable_target_contexts,-1,False
1,ADHD,2014,lived_only,310,310,296,296,243,209,291,5,additudemag.com,0.074324,0,14,0.0,False,all_markable_target_contexts,-1,False
2,ADHD,2014,mixed,183,183,173,173,152,133,170,3,rrstar.com,0.034682,0,10,0.0,False,all_markable_target_contexts,-1,False
3,ADHD,2014,substantive_core_overall,1779,1779,1689,1689,1093,874,1657,32,additudemag.com,0.024867,0,90,0.0,False,all_markable_target_contexts,-1,False
4,ADHD,2015,clinical_only,1203,1203,1130,1130,774,586,1101,29,rightdiagnosis.com,0.025664,0,73,0.0,False,all_markable_target_contexts,-1,False
5,ADHD,2015,lived_only,216,216,204,204,169,151,199,5,flinger.us,0.039216,0,12,0.0,False,all_markable_target_contexts,-1,False
6,ADHD,2015,mixed,128,128,119,119,109,99,113,6,flinger.us,0.050420,0,9,0.0,False,all_markable_target_contexts,-1,False
7,ADHD,2015,substantive_core_overall,1547,1547,1453,1453,993,756,1413,40,rightdiagnosis.com,0.019959,0,94,0.0,False,all_markable_target_contexts,-1,False
8,ADHD,2016,clinical_only,1149,1149,1097,1097,790,662,1065,32,wellesley.edu,0.023701,0,52,0.0,False,all_markable_target_contexts,-1,False
9,ADHD,2016,lived_only,309,309,294,294,245,229,283,11,additudemag.com,0.023810,0,15,0.0,False,all_markable_target_contexts,-1,False


## Encode XL-LEXEME Contexts

The local XL-LEXEME model is used through `torch` and `transformers` only when the saved embedding cache is missing, stale, or deliberately rebuilt. Identical marked contexts are encoded once, saved under `data/interim/lsc/breadth/`, and checked with a manifest fingerprint so ordinary notebook reruns can reuse the expensive target-token embeddings without retraining or re-encoding.


In [ ]:
embedding_path = INTERIM_DIR / "lsc_breadth_embeddings.npy"
embedding_normalised_path = INTERIM_DIR / "lsc_breadth_embeddings_normalised.npy"
embedding_index_path = INTERIM_DIR / "lsc_breadth_embedding_index.csv"
unique_contexts_path = INTERIM_DIR / "lsc_breadth_unique_contexts.parquet"
embedding_manifest_path = INTERIM_DIR / "lsc_breadth_embedding_manifest.json"


unique_contexts = sampled_contexts[["marked_context"]].drop_duplicates().reset_index(drop=True)
unique_contexts["embedding_row_id"] = np.arange(len(unique_contexts), dtype=int)
current_marked_contexts = unique_contexts["marked_context"].tolist()


def fingerprint_marked_contexts(marked_contexts: list[str]) -> str:
    digest = hashlib.sha256()
    for marked_context in marked_contexts:
        digest.update(marked_context.encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()


def build_embedding_manifest(marked_contexts: list[str], embedding_dimensions: int | None = None) -> dict[str, object]:
    return {
        "cache_schema_version": 1,
        "context_fingerprint": fingerprint_marked_contexts(marked_contexts),
        "unique_embedding_rows": len(marked_contexts),
        "embedding_dimensions": embedding_dimensions,
        "model_path": str(XL_LEXEME_MODEL_PATH.relative_to(PROJECT_ROOT)),
        "max_sequence_length": MAX_SEQUENCE_LENGTH,
        "target_start_marker": TARGET_START,
        "target_end_marker": TARGET_END,
    }


def manifest_matches_current_cache(manifest: dict[str, object], expected_manifest: dict[str, object]) -> bool:
    checked_keys = [
        "cache_schema_version",
        "context_fingerprint",
        "unique_embedding_rows",
        "model_path",
        "max_sequence_length",
        "target_start_marker",
        "target_end_marker",
    ]
    return all(manifest.get(key) == expected_manifest.get(key) for key in checked_keys)


def validate_loaded_embeddings(
    loaded_embeddings: np.ndarray,
    loaded_embeddings_normalised: np.ndarray,
    loaded_unique_contexts: pd.DataFrame,
) -> None:
    expected_rows = len(loaded_unique_contexts)
    if loaded_embeddings.ndim != 2 or loaded_embeddings_normalised.ndim != 2:
        raise RuntimeError("Cached XL-LEXEME embeddings must be two-dimensional arrays.")
    if loaded_embeddings.shape != loaded_embeddings_normalised.shape:
        raise RuntimeError("Cached raw and normalised XL-LEXEME embeddings have different shapes.")
    if loaded_embeddings.shape[0] != expected_rows:
        raise RuntimeError("Cached XL-LEXEME embedding row count does not match unique contexts.")
    if not loaded_unique_contexts["embedding_row_id"].equals(pd.Series(np.arange(expected_rows), name="embedding_row_id")):
        raise RuntimeError("Cached XL-LEXEME embedding row IDs are not contiguous from zero.")
    if loaded_unique_contexts["marked_context"].tolist() != current_marked_contexts:
        raise RuntimeError("Cached XL-LEXEME contexts do not match the current marked-context order.")
    normalised_norms = np.linalg.norm(loaded_embeddings_normalised, axis=1)
    if not np.all(np.isfinite(loaded_embeddings)) or not np.all(np.isfinite(loaded_embeddings_normalised)):
        raise RuntimeError("Cached XL-LEXEME embeddings contain non-finite values.")
    if not np.allclose(normalised_norms, 1.0, atol=1e-4):
        raise RuntimeError("Cached normalised XL-LEXEME embeddings are not unit length.")


def load_embedding_cache() -> tuple[np.ndarray, np.ndarray, pd.DataFrame, str] | None:
    if not embedding_path.exists() or not embedding_normalised_path.exists():
        return None

    expected_manifest = build_embedding_manifest(current_marked_contexts)
    if embedding_manifest_path.exists() and unique_contexts_path.exists():
        manifest = json.loads(embedding_manifest_path.read_text())
        if not manifest_matches_current_cache(manifest, expected_manifest):
            return None
        cached_unique_contexts = pd.read_parquet(unique_contexts_path)
        cache_status = "loaded_manifest_cache"
    elif sampled_contexts_path.exists():
        cached_sampled_contexts = pd.read_parquet(sampled_contexts_path)
        required_columns = {"marked_context", "embedding_row_id", "truncated_for_model"}
        if not required_columns.issubset(cached_sampled_contexts.columns):
            return None
        cached_unique_contexts = (
            cached_sampled_contexts[["marked_context", "embedding_row_id", "truncated_for_model"]]
            .drop_duplicates()
            .sort_values("embedding_row_id")
            .reset_index(drop=True)
        )
        if cached_unique_contexts["marked_context"].tolist() != current_marked_contexts:
            return None
        cache_status = "loaded_legacy_cache"
    else:
        return None

    loaded_embeddings = np.load(embedding_path)
    loaded_embeddings_normalised = np.load(embedding_normalised_path)
    cached_unique_contexts["embedding_row_id"] = cached_unique_contexts["embedding_row_id"].astype(int)
    cached_unique_contexts["truncated_for_model"] = cached_unique_contexts["truncated_for_model"].astype(bool)
    validate_loaded_embeddings(loaded_embeddings, loaded_embeddings_normalised, cached_unique_contexts)
    return loaded_embeddings, loaded_embeddings_normalised, cached_unique_contexts, cache_status


loaded_cache = None if REBUILD_XL_LEXEME_EMBEDDINGS else load_embedding_cache()
if loaded_cache is not None:
    embeddings, embeddings_normalised, unique_contexts, embedding_cache_status = loaded_cache
else:
    try:
        import torch
        from transformers import AutoModel, AutoTokenizer
    except ImportError as exc:
        raise ImportError(
            "Semantic breadth requires torch and transformers. Install/update the msc-nlp environment from environment.yml."
        ) from exc

    torch.set_num_threads(2)
    tokenizer = AutoTokenizer.from_pretrained(str(XL_LEXEME_MODEL_PATH), local_files_only=True)
    tokenizer.model_max_length = 100_000
    model = AutoModel.from_pretrained(str(XL_LEXEME_MODEL_PATH), local_files_only=True).to(DEVICE)
    model.eval()

    start_marker_id = tokenizer.convert_tokens_to_ids(TARGET_START)
    end_marker_id = tokenizer.convert_tokens_to_ids(TARGET_END)
    if start_marker_id == tokenizer.unk_token_id or end_marker_id == tokenizer.unk_token_id:
        raise RuntimeError("XL-LEXEME tokenizer does not recognise target marker tokens <t> and </t>.")

    def find_marker_pair(token_ids: list[int]) -> tuple[int, int]:
        try:
            start = token_ids.index(start_marker_id)
            end = token_ids.index(end_marker_id, start + 1)
        except ValueError as exc:
            raise RuntimeError("Tokenised context is missing XL-LEXEME target markers.") from exc
        if end <= start + 1:
            raise RuntimeError("XL-LEXEME target markers contain no target tokens.")
        return start, end

    def build_model_inputs(marked_context: str) -> tuple[list[int], list[int], bool]:
        token_ids = tokenizer.encode(marked_context, add_special_tokens=False)
        start, end = find_marker_pair(token_ids)
        payload_max = MAX_SEQUENCE_LENGTH - 2
        truncated = len(token_ids) > payload_max
        if truncated:
            target_center = (start + end) // 2
            window_start = max(0, target_center - payload_max // 2)
            window_start = min(window_start, max(0, len(token_ids) - payload_max))
            if start < window_start:
                window_start = max(0, start - 1)
            if end >= window_start + payload_max:
                window_start = max(0, end - payload_max + 1)
            window_end = min(len(token_ids), window_start + payload_max)
            token_ids = token_ids[window_start:window_end]
            start, end = find_marker_pair(token_ids)

        input_ids = [tokenizer.cls_token_id] + token_ids + [tokenizer.sep_token_id]
        target_mask = [0] * len(input_ids)
        for position in range(start + 2, end + 1):
            target_mask[position] = 1
        return input_ids, target_mask, truncated

    def encode_marked_contexts(marked_texts: list[str]) -> tuple[np.ndarray, np.ndarray]:
        embeddings = []
        truncated_flags = []
        for start in tqdm(range(0, len(marked_texts), ENCODE_BATCH_SIZE), desc="Encoding XL-LEXEME contexts"):
            batch_texts = marked_texts[start : start + ENCODE_BATCH_SIZE]
            built = [build_model_inputs(text) for text in batch_texts]
            batch_input_ids, batch_target_masks, batch_truncated = zip(*built)
            max_len = max(len(ids) for ids in batch_input_ids)
            padded_ids = []
            attention_masks = []
            padded_target_masks = []
            for input_ids, target_mask in zip(batch_input_ids, batch_target_masks):
                pad_len = max_len - len(input_ids)
                padded_ids.append(input_ids + [tokenizer.pad_token_id] * pad_len)
                attention_masks.append([1] * len(input_ids) + [0] * pad_len)
                padded_target_masks.append(target_mask + [0] * pad_len)
            input_tensor = torch.tensor(padded_ids, dtype=torch.long, device=DEVICE)
            attention_tensor = torch.tensor(attention_masks, dtype=torch.long, device=DEVICE)
            target_mask_tensor = torch.tensor(padded_target_masks, dtype=torch.bool, device=DEVICE)
            with torch.no_grad():
                hidden = model(input_ids=input_tensor, attention_mask=attention_tensor).last_hidden_state
            for row_index in range(hidden.shape[0]):
                target_hidden = hidden[row_index][target_mask_tensor[row_index]]
                embeddings.append(target_hidden.mean(dim=0).cpu().numpy().astype("float32"))
            truncated_flags.extend(batch_truncated)
        return np.vstack(embeddings).astype("float32"), np.array(truncated_flags, dtype=bool)

    embeddings, truncated_for_model = encode_marked_contexts(current_marked_contexts)
    embedding_norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    if np.any(embedding_norms == 0):
        raise RuntimeError("One or more XL-LEXEME embeddings has zero norm.")
    embeddings_normalised = (embeddings / embedding_norms).astype("float32")
    unique_contexts["truncated_for_model"] = truncated_for_model
    embedding_cache_status = "rebuilt_cache" if REBUILD_XL_LEXEME_EMBEDDINGS else "created_cache"
    np.save(embedding_path, embeddings)
    np.save(embedding_normalised_path, embeddings_normalised)

unique_contexts.to_parquet(unique_contexts_path, index=False)
embedding_manifest = build_embedding_manifest(current_marked_contexts, embedding_dimensions=int(embeddings.shape[1]))
embedding_manifest_path.write_text(json.dumps(embedding_manifest, indent=2, sort_keys=True) + "\n")

sampled_contexts = sampled_contexts.merge(unique_contexts, on="marked_context", how="left")
if sampled_contexts["embedding_row_id"].isna().any():
    raise RuntimeError("Some sampled contexts did not receive an embedding row id.")
sampled_contexts["embedding_row_id"] = sampled_contexts["embedding_row_id"].astype(int)
sampled_contexts["truncated_for_model"] = sampled_contexts["truncated_for_model"].astype(bool)
truncated_for_model = unique_contexts["truncated_for_model"].to_numpy(dtype=bool)

embedding_index = sampled_contexts[
    [
        "sample_row_id",
        "embedding_row_id",
        "context_row_id",
        "source_context_row_id",
        "context_id",
        "doc_id",
        "lsc_year",
        "analysis_unit",
        "frame_stratum",
        "term_role",
        "target_group",
        "raw_form",
        "registered_domain",
        "context_source",
        "context_token_count",
        "truncated_for_model",
    ]
].copy()

embedding_index.to_csv(embedding_index_path, index=False)
sampled_contexts.to_parquet(sampled_contexts_path, index=False)

pd.DataFrame(
    {
        "metric": [
            "embedding_cache_status",
            "sampled_context_rows",
            "unique_embedding_rows",
            "embedding_dimensions",
            "contexts_truncated_for_model",
        ],
        "value": [
            embedding_cache_status,
            len(sampled_contexts),
            embeddings.shape[0],
            embeddings.shape[1],
            int(truncated_for_model.sum()),
        ],
    }
)


## Annual Breadth Scores

Breadth is the mean pairwise cosine distance within each unit-year-frame sample. The calculation uses L2-normalised embeddings and a closed-form expression, avoiding materialising all pairwise distances for the main score.


In [6]:
def mean_pairwise_cosine_distance(normalised_vectors: np.ndarray) -> float:
    n = normalised_vectors.shape[0]
    if n < 2:
        return float("nan")
    sum_vector = normalised_vectors.sum(axis=0)
    sum_pairwise_similarity = (float(np.dot(sum_vector, sum_vector)) - n) / 2.0
    pair_count = n * (n - 1) / 2.0
    mean_similarity = sum_pairwise_similarity / pair_count
    return float(1.0 - mean_similarity)


def bootstrap_document_breadth(group_index: pd.DataFrame, normalised_vectors: np.ndarray, rng: np.random.Generator) -> dict[str, object]:
    document_to_rows = [values.to_numpy(dtype=int) for _, values in group_index.groupby("doc_id")["embedding_row_id"]]
    if len(document_to_rows) < 2:
        return {
            "bootstrap_repetitions": 0,
            "bootstrap_unit": "doc_id",
            "breadth_bootstrap_mean": float("nan"),
            "breadth_ci_low": float("nan"),
            "breadth_ci_high": float("nan"),
        }
    bootstrap_values = []
    for _ in range(BOOTSTRAP_REPETITIONS):
        sampled_docs = rng.integers(0, len(document_to_rows), len(document_to_rows))
        sampled_rows = np.concatenate([document_to_rows[index] for index in sampled_docs])
        if len(sampled_rows) >= 2:
            bootstrap_values.append(mean_pairwise_cosine_distance(normalised_vectors[sampled_rows]))
    return {
        "bootstrap_repetitions": BOOTSTRAP_REPETITIONS,
        "bootstrap_unit": "doc_id",
        "breadth_bootstrap_mean": float(np.mean(bootstrap_values)),
        "breadth_ci_low": float(np.quantile(bootstrap_values, 0.025)),
        "breadth_ci_high": float(np.quantile(bootstrap_values, 0.975)),
    }


breadth_records = []
for (unit, year, frame_stratum), group in embedding_index.groupby(["analysis_unit", "lsc_year", "frame_stratum"], sort=True):
    row_ids = group["embedding_row_id"].to_numpy(dtype=int)
    vectors = embeddings_normalised[row_ids]
    metadata = group.iloc[0]
    bootstrap = bootstrap_document_breadth(group, embeddings_normalised, BOOTSTRAP_RNG)
    breadth_records.append(
        {
            "lsc_year": int(year),
            "analysis_unit": unit,
            "frame_stratum": frame_stratum,
            "term_role": metadata["term_role"],
            "target_group": metadata["target_group"],
            "breadth_mean_pairwise_cosine_distance": mean_pairwise_cosine_distance(vectors),
            "sampled_contexts": int(len(group)),
            "sampled_documents": int(group["doc_id"].nunique()),
            "sampled_domains": int(group["registered_domain"].nunique()),
            "contexts_truncated_for_model": int(group["truncated_for_model"].sum()),
            **bootstrap,
        }
    )

annual_breadth = pd.DataFrame(breadth_records).sort_values(["analysis_unit", "frame_stratum", "lsc_year"]).reset_index(drop=True)
annual_breadth = annual_breadth.merge(
    sampling_diagnostics[
        [
            "analysis_unit",
            "lsc_year",
            "frame_stratum",
            "available_contexts",
            "markable_contexts",
            "unmarkable_contexts",
            "unmarkable_share",
            "cap_applied",
            "sampling_policy",
            "top_domain",
            "top_domain_share",
            "max_contexts_per_unit_year_frame",
            "small_cell_flag",
        ]
    ],
    on=["analysis_unit", "lsc_year", "frame_stratum"],
    how="left",
)

annual_breadth_path = PROCESSED_DIR / "lsc_breadth_annual_scores.csv"
annual_breadth.to_csv(annual_breadth_path, index=False)
annual_breadth.head(12)


,lsc_year,analysis_unit,frame_stratum,term_role,target_group,breadth_mean_pairwise_cosine_distance,sampled_contexts,sampled_documents,sampled_domains,contexts_truncated_for_model,bootstrap_repetitions,bootstrap_unit,breadth_bootstrap_mean,breadth_ci_low,breadth_ci_high,available_contexts,markable_contexts,unmarkable_contexts,unmarkable_share,cap_applied,sampling_policy,top_domain,top_domain_share,max_contexts_per_unit_year_frame,small_cell_flag
0,2014,ADHD,clinical_only,target,ADHD,0.150406,1220,808,663,39,500,doc_id,0.150336,0.144180,0.156634,1286,1220,0,0.0,False,all_markable_target_contexts,naturalnews.com,0.021311,-1,False
1,2015,ADHD,clinical_only,target,ADHD,0.153624,1130,774,586,40,500,doc_id,0.153277,0.145953,0.160045,1203,1130,0,0.0,False,all_markable_target_contexts,rightdiagnosis.com,0.025664,-1,False
2,2016,ADHD,clinical_only,target,ADHD,0.153201,1097,790,662,36,500,doc_id,0.153152,0.145962,0.160676,1149,1097,0,0.0,False,all_markable_target_contexts,wellesley.edu,0.023701,-1,False
3,2017,ADHD,clinical_only,target,ADHD,0.143280,1048,751,655,27,500,doc_id,0.143096,0.135722,0.149897,1099,1048,0,0.0,False,all_markable_target_contexts,rehabs.com,0.018130,-1,False
4,2018,ADHD,clinical_only,target,ADHD,0.139570,1128,787,719,29,500,doc_id,0.139291,0.132282,0.146066,1182,1128,0,0.0,False,all_markable_target_contexts,bedford.gov.uk,0.015071,-1,False
5,2019,ADHD,clinical_only,target,ADHD,0.137691,950,674,619,36,500,doc_id,0.137729,0.129888,0.145802,1000,950,0,0.0,False,all_markable_target_contexts,flavio.pl,0.009474,-1,False
6,2020,ADHD,clinical_only,target,ADHD,0.136097,918,668,622,28,500,doc_id,0.136064,0.128067,0.143475,959,918,0,0.0,False,all_markable_target_contexts,digication.com,0.006536,-1,False
7,2021,ADHD,clinical_only,target,ADHD,0.146475,890,636,579,43,500,doc_id,0.146208,0.138735,0.153631,928,890,0,0.0,False,all_markable_target_contexts,suffolkfamilytherapy.com,0.012360,-1,False
8,2022,ADHD,clinical_only,target,ADHD,0.138081,881,634,595,29,500,doc_id,0.138148,0.130118,0.146300,930,881,0,0.0,False,all_markable_target_contexts,thestudiolife.com,0.013621,-1,False
9,2023,ADHD,clinical_only,target,ADHD,0.140711,719,477,446,18,500,doc_id,0.140443,0.132044,0.148195,750,719,0,0.0,False,all_markable_target_contexts,turningwinds.com,0.023644,-1,False


## Trend Models And Audit Flags

Each annual breadth series gets the same compact Baes-style trend summary used for the other scalar measures. Audit flags identify sparse frame-year samples, high domain concentration, extensive truncation, and residual autocorrelation in the trend model.


In [7]:
def fit_trend(frame: pd.DataFrame, value_column: str) -> dict[str, object]:
    data = frame[["lsc_year", value_column]].dropna().sort_values("lsc_year")
    if len(data) < 3 or data[value_column].nunique() < 2:
        return {
            "n_years": len(data),
            "year_center": np.nan,
            "linear_intercept": np.nan,
            "linear_slope_per_year": np.nan,
            "linear_slope_se": np.nan,
            "linear_p_value": np.nan,
            "linear_r_squared": np.nan,
            "linear_adj_r_squared": np.nan,
            "standardized_beta_year": np.nan,
            "durbin_watson": np.nan,
            "lag1_residual_autocorrelation": np.nan,
            "autocorrelation_flag": False,
            "ar1_sensitivity_slope_per_year": np.nan,
            "ar1_sensitivity_p_value": np.nan,
            "quadratic_adj_r_squared": np.nan,
            "quadratic_delta_adj_r_squared": np.nan,
        }
    years = data["lsc_year"].to_numpy(dtype=float)
    values = data[value_column].to_numpy(dtype=float)
    year_center = float(years.mean())
    x = years - year_center
    result = stats.linregress(x, values)
    fitted = result.intercept + result.slope * x
    residuals = values - fitted
    sse = float(np.sum(residuals**2))
    sst = float(np.sum((values - values.mean()) ** 2))
    r_squared = 1 - sse / sst if sst else np.nan
    n = len(values)
    adj_r_squared = 1 - (1 - r_squared) * (n - 1) / (n - 2) if n > 2 and not np.isnan(r_squared) else np.nan
    std_beta = result.slope * np.std(x, ddof=1) / np.std(values, ddof=1) if np.std(values, ddof=1) else np.nan
    dw_denominator = float(np.sum(residuals**2))
    durbin_watson = float(np.sum(np.diff(residuals) ** 2) / dw_denominator) if dw_denominator else np.nan
    lag1 = float(np.corrcoef(residuals[:-1], residuals[1:])[0, 1]) if n >= 4 and np.std(residuals) else np.nan
    autocorrelation_flag = bool(
        not np.isnan(durbin_watson)
        and (durbin_watson < DW_AUTOCORRELATION_LOW or durbin_watson > DW_AUTOCORRELATION_HIGH)
    )
    ar1_slope = np.nan
    ar1_p_value = np.nan
    if autocorrelation_flag and n >= 5 and not np.isnan(lag1) and abs(lag1) < 0.98:
        y_star = values[1:] - lag1 * values[:-1]
        x_star = x[1:] - lag1 * x[:-1]
        ar1_result = stats.linregress(x_star, y_star)
        ar1_slope = float(ar1_result.slope)
        ar1_p_value = float(ar1_result.pvalue)
    quadratic_adj_r_squared = np.nan
    quadratic_delta = np.nan
    if n >= 6:
        q_coefficients = np.polyfit(x, values, deg=2)
        q_fitted = np.polyval(q_coefficients, x)
        q_sse = float(np.sum((values - q_fitted) ** 2))
        q_r_squared = 1 - q_sse / sst if sst else np.nan
        quadratic_adj_r_squared = 1 - (1 - q_r_squared) * (n - 1) / (n - 3) if n > 3 and not np.isnan(q_r_squared) else np.nan
        quadratic_delta = quadratic_adj_r_squared - adj_r_squared if not np.isnan(quadratic_adj_r_squared) else np.nan
    return {
        "n_years": n,
        "year_center": year_center,
        "linear_intercept": float(result.intercept),
        "linear_slope_per_year": float(result.slope),
        "linear_slope_se": float(result.stderr) if result.stderr is not None else np.nan,
        "linear_p_value": float(result.pvalue),
        "linear_r_squared": float(r_squared),
        "linear_adj_r_squared": float(adj_r_squared),
        "standardized_beta_year": float(std_beta),
        "durbin_watson": durbin_watson,
        "lag1_residual_autocorrelation": lag1,
        "autocorrelation_flag": autocorrelation_flag,
        "ar1_sensitivity_slope_per_year": ar1_slope,
        "ar1_sensitivity_p_value": ar1_p_value,
        "quadratic_adj_r_squared": quadratic_adj_r_squared,
        "quadratic_delta_adj_r_squared": quadratic_delta,
    }


trend_rows = []
for group_values, frame in annual_breadth.groupby(["analysis_unit", "frame_stratum", "term_role", "target_group"], sort=True):
    analysis_unit, frame_stratum, term_role, target_group = group_values
    trend_rows.append(
        {
            "analysis_unit": analysis_unit,
            "frame_stratum": frame_stratum,
            "term_role": term_role,
            "target_group": target_group,
            "index_name": "breadth_mean_pairwise_cosine_distance",
            **fit_trend(frame, "breadth_mean_pairwise_cosine_distance"),
        }
    )
trend_summary = pd.DataFrame(trend_rows)
trend_summary_path = PROCESSED_DIR / "lsc_breadth_trend_models.csv"
trend_summary.to_csv(trend_summary_path, index=False)

flag_rows = []
for row in annual_breadth.itertuples(index=False):
    flags = []
    if bool(row.small_cell_flag):
        flags.append("small_frame_year_cell")
    if row.sampled_contexts < 100:
        flags.append("sampled_contexts_lt_100")
    if row.sampled_documents < 50:
        flags.append("sampled_documents_lt_50")
    if row.unmarkable_share > 0.05:
        flags.append("unmarkable_share_gt_0_05")
    if pd.notna(row.top_domain_share) and row.top_domain_share > 0.20:
        flags.append("top_domain_share_gt_0_20")
    if row.contexts_truncated_for_model / max(row.sampled_contexts, 1) > 0.50:
        flags.append("truncated_context_share_gt_0_50")
    if flags:
        flag_rows.append(
            {
                "lsc_year": row.lsc_year,
                "analysis_unit": row.analysis_unit,
                "frame_stratum": row.frame_stratum,
                "flags": ";".join(flags),
                "sampled_contexts": row.sampled_contexts,
                "sampled_documents": row.sampled_documents,
                "unmarkable_share": row.unmarkable_share,
                "top_domain_share": row.top_domain_share,
                "contexts_truncated_for_model": row.contexts_truncated_for_model,
            }
        )

for row in trend_summary.loc[trend_summary["autocorrelation_flag"]].itertuples(index=False):
    flag_rows.append(
        {
            "lsc_year": pd.NA,
            "analysis_unit": row.analysis_unit,
            "frame_stratum": row.frame_stratum,
            "flags": "trend_residual_autocorrelation",
            "sampled_contexts": pd.NA,
            "sampled_documents": pd.NA,
            "unmarkable_share": pd.NA,
            "top_domain_share": pd.NA,
            "contexts_truncated_for_model": pd.NA,
        }
    )

audit_flag_columns = [
    "lsc_year",
    "analysis_unit",
    "frame_stratum",
    "flags",
    "sampled_contexts",
    "sampled_documents",
    "unmarkable_share",
    "top_domain_share",
    "contexts_truncated_for_model",
]
audit_flags = pd.DataFrame(flag_rows, columns=audit_flag_columns)
audit_flags_path = PROCESSED_DIR / "lsc_breadth_audit_flags.csv"
audit_flags.to_csv(audit_flags_path, index=False)

trend_summary.head(12)


,analysis_unit,frame_stratum,term_role,target_group,index_name,n_years,year_center,linear_intercept,linear_slope_per_year,linear_slope_se,linear_p_value,linear_r_squared,linear_adj_r_squared,standardized_beta_year,durbin_watson,lag1_residual_autocorrelation,autocorrelation_flag,ar1_sensitivity_slope_per_year,ar1_sensitivity_p_value,quadratic_adj_r_squared,quadratic_delta_adj_r_squared
0,ADHD,clinical_only,target,ADHD,breadth_mean_pairwise_cosine_distance,13,2020.0,0.140938,-0.001787,0.000337,0.000251,0.718961,0.693412,-0.847916,1.658145,0.168060,False,NaN,NaN,0.664325,-0.029087
1,ADHD,lived_only,target,ADHD,breadth_mean_pairwise_cosine_distance,13,2020.0,0.079186,-0.000366,0.000610,0.560273,0.031752,-0.056270,-0.178192,3.205371,-0.607714,True,-0.000437,0.246934,-0.161897,-0.105627
2,ADHD,mixed,target,ADHD,breadth_mean_pairwise_cosine_distance,13,2020.0,0.092332,-0.001392,0.000654,0.056568,0.292031,0.227670,-0.540399,2.184142,-0.201953,False,NaN,NaN,0.202263,-0.025407
3,ADHD,substantive_core_overall,target,ADHD,breadth_mean_pairwise_cosine_distance,13,2020.0,0.126990,-0.002037,0.000313,0.000044,0.793506,0.774734,-0.890789,2.456148,-0.242392,False,NaN,NaN,0.771752,-0.002982
4,Autism,clinical_only,target,Autism,breadth_mean_pairwise_cosine_distance,13,2020.0,0.093488,0.000749,0.000315,0.036319,0.340444,0.280485,0.583476,0.960795,0.393735,True,0.000157,0.754792,0.789346,0.508861
5,Autism,lived_only,target,Autism,breadth_mean_pairwise_cosine_distance,13,2020.0,0.100141,-0.001040,0.000416,0.029581,0.362056,0.304061,-0.601711,1.514107,0.240805,False,NaN,NaN,0.460364,0.156303
6,Autism,mixed,target,Autism,breadth_mean_pairwise_cosine_distance,13,2020.0,0.097688,-0.000294,0.000570,0.616313,0.023601,-0.065163,-0.153625,1.692841,-0.179926,False,NaN,NaN,0.219844,0.285007
7,Autism,substantive_core_overall,target,Autism,breadth_mean_pairwise_cosine_distance,13,2020.0,0.099261,0.000119,0.000245,0.637442,0.020906,-0.068103,0.144587,1.175914,0.379361,True,-0.000133,0.752659,0.510775,0.578878
8,frustration,unframed_baseline,baseline,baseline,breadth_mean_pairwise_cosine_distance,13,2020.0,0.092264,0.001951,0.000560,0.005102,0.524762,0.481559,0.724405,1.729714,0.066841,False,NaN,NaN,0.545727,0.064168
9,loneliness,unframed_baseline,baseline,baseline,breadth_mean_pairwise_cosine_distance,13,2020.0,0.069130,-0.000719,0.000367,0.075559,0.259227,0.191884,-0.509144,1.952140,-0.055654,False,NaN,NaN,0.194384,0.002499


## Breadth Trajectories

The report-facing trajectory figure uses three equal-width panels: ADHD, Autism, and comparator terms. The ADHD and Autism panels foreground the substantive-core Overall trajectory and add clinical/disorder and lived-experience traces as lighter contextual lines.

Mixed-frame estimates, sampling diagnostics, raw-form balance, small-cell warnings, and trend diagnostics remain available in the saved CSV tables and audit flags. They are not saved as separate report figures in order to keep the figure folder aligned with the main dissertation story.


In [ ]:
READER_FRAME_STRATA = ["clinical_only", "lived_only"]
READER_FRAME_LABELS = {
    "clinical_only": "Clinical/disorder framing",
    "lived_only": "Lived-experience framing",
}
BREADTH_COLUMN = "breadth_mean_pairwise_cosine_distance"


def save_lsc_figure(fig: plt.Figure, png_path: Path) -> Path:
    fig.tight_layout(pad=1.1, rect=[0, 0, 1, 0.90])
    fig.savefig(png_path, dpi=LSC_FIGURE_DPI, bbox_inches="tight", facecolor="white")
    pdf_path = png_path.with_suffix(".pdf")
    fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    return pdf_path


def series_color(unit: str, frame_stratum: str) -> str:
    if unit in CONDITION_FRAME_COLORS and frame_stratum in CONDITION_FRAME_COLORS[unit]:
        return CONDITION_FRAME_COLORS[unit][frame_stratum]
    if unit in UNIT_COLOURS:
        return UNIT_COLOURS[unit]
    return FRAME_COLORS.get(frame_stratum, "#7B8785")


def trend_line_for(series: pd.DataFrame, trend: pd.Series) -> tuple[np.ndarray, np.ndarray]:
    years = series["lsc_year"].to_numpy(dtype=float)
    fitted = trend["linear_intercept"] + trend["linear_slope_per_year"] * (years - trend["year_center"])
    return years, fitted


def trend_for(unit: str, frame_stratum: str) -> pd.Series | None:
    row = trend_summary.loc[
        trend_summary["analysis_unit"].eq(unit) & trend_summary["frame_stratum"].eq(frame_stratum)
    ]
    if row.empty or pd.isna(row.iloc[0]["linear_slope_per_year"]):
        return None
    return row.iloc[0]


def y_limits_from(frame: pd.DataFrame, value_column: str, ci_low: str | None = None, ci_high: str | None = None) -> tuple[float, float]:
    values = [frame[value_column].to_numpy(dtype=float)]
    if ci_low and ci_low in frame:
        values.append(frame[ci_low].to_numpy(dtype=float))
    if ci_high and ci_high in frame:
        values.append(frame[ci_high].to_numpy(dtype=float))
    finite_values = [v[np.isfinite(v)] for v in values if len(v)]
    combined = np.concatenate(finite_values)
    low, high = float(combined.min()), float(combined.max())
    padding = max((high - low) * 0.10, 0.004)
    return low - padding, high + padding


def style_year_axis(ax: plt.Axes) -> None:
    ax.set_xticks(EXPECTED_YEARS[::2])
    ax.tick_params(axis="x", labelsize=8.5)


def plot_line_with_trend(
    ax: plt.Axes,
    frame: pd.DataFrame,
    unit: str,
    frame_stratum: str,
    value_column: str,
    color: str,
    marker: str,
    label: str | None = None,
    ci_low: str | None = None,
    ci_high: str | None = None,
    ribbon_alpha: float = 0.12,
    linewidth: float = 2.2,
    markersize: float = 4.8,
    alpha: float = 1.0,
    show_trend: bool = True,
) -> None:
    series = frame.loc[frame["analysis_unit"].eq(unit) & frame["frame_stratum"].eq(frame_stratum)].sort_values("lsc_year")
    if series.empty:
        return
    ax.plot(
        series["lsc_year"],
        series[value_column],
        marker=marker,
        markersize=markersize,
        linewidth=linewidth,
        label=label,
        color=color,
        alpha=alpha,
    )
    if ci_low and ci_high:
        ax.fill_between(
            series["lsc_year"].to_numpy(dtype=float),
            series[ci_low].to_numpy(dtype=float),
            series[ci_high].to_numpy(dtype=float),
            color=color,
            alpha=ribbon_alpha,
            linewidth=0,
        )
    trend = trend_for(unit, frame_stratum)
    if show_trend and trend is not None:
        years, fitted = trend_line_for(series, trend)
        ax.plot(years, fitted, color=color, linewidth=1.05, linestyle="--", alpha=min(alpha + 0.12, 0.92))


def plot_target_panel(ax: plt.Axes, unit: str) -> None:
    plot_line_with_trend(
        ax,
        annual_breadth,
        unit,
        "substantive_core_overall",
        BREADTH_COLUMN,
        series_color(unit, "substantive_core_overall"),
        FRAME_MARKERS["substantive_core_overall"],
        "Overall",
        "breadth_ci_low",
        "breadth_ci_high",
        ribbon_alpha=0.14,
        linewidth=2.8,
        markersize=4.8,
    )
    for frame_stratum in READER_FRAME_STRATA:
        plot_line_with_trend(
            ax,
            annual_breadth,
            unit,
            frame_stratum,
            BREADTH_COLUMN,
            series_color(unit, frame_stratum),
            FRAME_MARKERS[frame_stratum],
            READER_FRAME_LABELS[frame_stratum],
            "breadth_ci_low",
            "breadth_ci_high",
            ribbon_alpha=0.075,
            linewidth=1.55,
            markersize=3.7,
            alpha=0.82,
        )
    ax.set_title(unit, loc="left", fontsize=12, fontweight="bold")
    style_year_axis(ax)
    ax.legend(loc="best", fontsize=7.6)


def plot_baseline_panel(
    ax: plt.Axes,
    frame: pd.DataFrame,
    value_column: str,
    ci_low: str | None = None,
    ci_high: str | None = None,
    show_trend: bool = True,
) -> None:
    for unit in BASELINE_UNITS:
        plot_line_with_trend(
            ax,
            frame,
            unit,
            BASELINE_FRAME_STRATUM,
            value_column,
            UNIT_COLOURS[unit],
            UNIT_MARKERS[unit] if "UNIT_MARKERS" in globals() else LSC_UNIT_MARKERS[unit],
            LSC_UNIT_LABELS[unit] if "LSC_UNIT_LABELS" in globals() else unit,
            ci_low,
            ci_high,
            ribbon_alpha=0.08,
            linewidth=2.0,
            show_trend=show_trend,
        )
    ax.set_title("Comparator terms", loc="left", fontsize=12, fontweight="bold")
    ax.legend(loc="best", fontsize=7.6)
    style_year_axis(ax)


main_rows = pd.concat(
    [
        annual_breadth.loc[
            annual_breadth["analysis_unit"].isin(TARGET_UNITS)
            & annual_breadth["frame_stratum"].isin(["substantive_core_overall", *READER_FRAME_STRATA])
        ],
        annual_breadth.loc[annual_breadth["frame_stratum"].eq(BASELINE_FRAME_STRATUM)],
    ],
    ignore_index=True,
)
main_ylim = y_limits_from(main_rows, BREADTH_COLUMN, "breadth_ci_low", "breadth_ci_high")

trajectory_png = FIGURE_DIR / "lsc_breadth_trajectories.png"
fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.35), sharex=True, sharey=True)
fig.suptitle("Semantic breadth of target-use contexts", fontsize=15, fontweight="bold", x=0.02, ha="left")
fig.text(0.02, 0.895, "Higher values indicate more varied local contexts. Shaded bands are 95% annual uncertainty intervals; dashed lines are OLS trend summaries.", fontsize=9.1)
for ax, unit in zip(axes[:2], TARGET_UNITS):
    plot_target_panel(ax, unit)
plot_baseline_panel(
    axes[2],
    annual_breadth.loc[annual_breadth["frame_stratum"].eq(BASELINE_FRAME_STRATUM)],
    BREADTH_COLUMN,
    "breadth_ci_low",
    "breadth_ci_high",
)
for ax in axes:
    ax.set_ylim(*main_ylim)
    ax.set_xlabel("Publication year")
axes[0].set_ylabel("Mean pairwise cosine distance")
trajectory_pdf = save_lsc_figure(fig, trajectory_png)
plt.close(fig)



## Handoff Summary

The main handoff for downstream synthesis is the annual breadth score table. Sampled contexts, embedding arrays, embedding index, sampling diagnostics, raw-form diagnostics, trend models, and audit flags make the embedding run reproducible and inspectable.


In [9]:
expected_annual_rows = len(EXPECTED_YEARS) * (len(BASELINE_UNITS) + len(TARGET_UNITS) * len(TARGET_FRAME_STRATA))
target_sampling = sampling_diagnostics.loc[sampling_diagnostics["analysis_unit"].isin(TARGET_UNITS)]
baseline_sampling = sampling_diagnostics.loc[sampling_diagnostics["analysis_unit"].isin(BASELINE_UNITS)]

summary = {
    "annual_rows": len(annual_breadth),
    "expected_annual_rows": expected_annual_rows,
    "sampled_context_rows": len(sampled_contexts),
    "unique_embedding_rows": int(embeddings.shape[0]),
    "embedding_index_rows": len(embedding_index),
    "embedding_dimensions": int(embeddings.shape[1]),
    "audit_flag_rows": len(audit_flags),
    "max_sampled_contexts_per_unit_year_frame": int(sampling_diagnostics["sampled_contexts"].max()),
    "max_sampled_target_contexts_per_unit_year_frame": int(target_sampling["sampled_contexts"].max()),
    "max_sampled_baseline_contexts_per_unit_year": int(baseline_sampling["sampled_contexts"].max()),
    "contexts_truncated_for_model": int(embedding_index["truncated_for_model"].sum()),
}
if summary["annual_rows"] != expected_annual_rows:
    raise RuntimeError(f"Expected {expected_annual_rows} annual rows, found {summary['annual_rows']}.")
if summary["unique_embedding_rows"] != len(unique_contexts):
    raise RuntimeError("Unique embedding row count does not match unique marked context count.")
if TARGET_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME is None:
    if not target_sampling["sampled_contexts"].eq(target_sampling["markable_contexts"]).all():
        raise RuntimeError("Target rows should be uncapped, but at least one target frame-year was sampled down.")
elif summary["max_sampled_target_contexts_per_unit_year_frame"] > TARGET_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME:
    raise RuntimeError("Target sampling cap was exceeded.")
if (
    BASELINE_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME is not None
    and summary["max_sampled_baseline_contexts_per_unit_year"] > BASELINE_MAX_CONTEXTS_PER_UNIT_YEAR_FRAME
):
    raise RuntimeError("Baseline sampling cap was exceeded.")
summary


{'annual_rows': 143,
 'expected_annual_rows': 143,
 'sampled_context_rows': 145430,
 'unique_embedding_rows': 86930,
 'embedding_index_rows': 145430,
 'embedding_dimensions': 1024,
 'audit_flag_rows': 5,
 'max_sampled_contexts_per_unit_year_frame': 4405,
 'max_sampled_target_contexts_per_unit_year_frame': 4405,
 'max_sampled_baseline_contexts_per_unit_year': 1000,
 'contexts_truncated_for_model': 3865}